# WAWA EMG Analysis
**Navid's STM32 EMG System — Recordings 8 & 9**

### Binary format (confirmed via parsing)
- Header: `0xDEADBEEF` (4 bytes, little-endian stored as `EFBEADDE`)
- Records: **45 bytes each**, at **500 Hz**
- Footer: `0xBEEFDEAD`

### Record layout (45 bytes)
| Offset | Type | Field |
|--------|------|-------|
| 0–3 | uint32 LE | Timestamp (ms) |
| 4–5 | uint16 LE | EMG Ch0 (RF) — 12-bit ADC |
| 6–7 | uint16 LE | EMG Ch1 (BF) — 12-bit ADC |
| 8–9 | uint16 LE | EMG Ch2 (Gas — cable too short, noisy) |
| 10–11 | uint16 LE | EMG Ch3 (TA — cable too short, noisy) |
| 12–13 | int16 LE | IMU Accel X |
| 14–15 | int16 LE | IMU Accel Y |
| 16–17 | int16 LE | IMU Accel Z |
| 18–19 | int16 LE | IMU Gyro X |
| 20–21 | int16 LE | IMU Gyro Y |
| 22–23 | int16 LE | IMU Gyro Z |
| 24–25 | int16 LE | IMU Mag X |
| 26–27 | int16 LE | IMU Mag Y |
| 28–29 | int16 LE | IMU Mag Z |
| 30–31 | int16 LE | (unknown/extra) |
| 32 | uint8 | Constant 0x3F (version?) |
| 33–44 | — | Zero padding |

### Recordings
- **Rec 8**: Stomp (MVC) → Walk (~51.7s total)
- **Rec 9**: Stomp (MVC) → Ramp up/down repeated (~68.6s total)

**Active channels**: Ch0 (RF) and Ch1 (BF) — Ch2/Ch3 cable too short, show noise/saturation artifacts

In [ ]:
import struct
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
FS = 500  # Hz

In [ ]:
def parse_emg_bin(path):
    """
    Parse Navid's STM32 EMG binary format.
    Returns dict with time (s), EMG channels (raw ADC), and IMU data.
    """
    with open(path, 'rb') as f:
        raw = f.read()

    HEADER = 0xDEADBEEF
    FOOTER = 0xBEEFDEAD
    RECORD_SIZE = 45

    header_val = struct.unpack('<I', raw[:4])[0]
    footer_val = struct.unpack('<I', raw[-4:])[0]
    assert header_val == HEADER, f"Bad header: {hex(header_val)}"
    assert footer_val == FOOTER, f"Bad footer: {hex(footer_val)}"

    payload = raw[4:-4]
    N = len(payload) // RECORD_SIZE
    remainder = len(payload) % RECORD_SIZE
    if remainder:
        print(f"  Warning: {remainder} leftover bytes ignored")

    # Preallocate
    ts   = np.empty(N, dtype=np.uint32)
    ch   = np.empty((N, 4), dtype=np.uint16)  # EMG Ch0-3 (raw ADC)
    acc  = np.empty((N, 3), dtype=np.int16)   # Accel XYZ
    gyr  = np.empty((N, 3), dtype=np.int16)   # Gyro XYZ
    mag  = np.empty((N, 3), dtype=np.int16)   # Mag XYZ

    fmt = '<I 4H 9h'
    struct_size = struct.calcsize(fmt)  # 4+8+18 = 30 bytes

    for i in range(N):
        rec = payload[i*RECORD_SIZE : i*RECORD_SIZE + struct_size]
        unpacked = struct.unpack(fmt, rec)
        ts[i]      = unpacked[0]
        ch[i]      = unpacked[1:5]
        acc[i]     = unpacked[5:8]
        gyr[i]     = unpacked[8:11]
        mag[i]     = unpacked[11:14]

    time_s = (ts - ts[0]) / 1000.0  # ms -> seconds

    print(f"  File: {Path(path).name}")
    print(f"  Records: {N} | Duration: {time_s[-1]:.1f}s | Fs: {FS} Hz")
    print(f"  Timestamp range: {ts[0]} – {ts[-1]} ms")

    return {
        'path': path,
        'time': time_s,
        'ch': ch,           # shape (N,4), uint16 ADC counts (0-4095)
        'acc': acc,         # shape (N,3), raw int16
        'gyr': gyr,         # shape (N,3), raw int16
        'mag': mag,         # shape (N,3), raw int16
        'N': N,
    }

print("Parsing Rec 8...")
rec8 = parse_emg_bin('EMG_rec_8.bin')
print("\nParsing Rec 9...")
rec9 = parse_emg_bin('EMG_rec_9.bin')

In [ ]:
def process_emg(raw_adc, fs=500, bandpass=(20, 200), smooth_ms=100):
    """
    Standard EMG processing pipeline:
      1. Demean (remove DC offset)
      2. Bandpass filter (20–200 Hz; upper limit < Nyquist=250 Hz at 500 Hz Fs)
      3. Full-wave rectify
      4. Linear envelope (low-pass, default 100ms smoothing)
    
    Returns: filtered, rectified, envelope (all same shape as input)
    """
    # 1. Demean
    x = raw_adc.astype(float) - raw_adc.mean()

    # 2. Bandpass (must be < Nyquist = fs/2)
    nyq = fs / 2
    b, a = signal.butter(4, [bandpass[0]/nyq, bandpass[1]/nyq], btype='band')
    filtered = signal.filtfilt(b, a, x)

    # 3. Rectify
    rectified = np.abs(filtered)

    # 4. Linear envelope via low-pass
    cutoff_hz = 1000 / smooth_ms  # e.g. 100ms -> 10 Hz
    b2, a2 = signal.butter(4, cutoff_hz / nyq, btype='low')
    envelope = signal.filtfilt(b2, a2, rectified)
    envelope = np.clip(envelope, 0, None)

    return filtered, rectified, envelope


def normalize_to_mvc(envelope, time_s, mvc_window=(0, 5)):
    """
    Normalize EMG envelope to MVC (max value in the stomping window).
    Returns %MVC (0-100+)
    """
    mask = (time_s >= mvc_window[0]) & (time_s <= mvc_window[1])
    mvc_val = np.percentile(envelope[mask], 95)  # 95th to reduce outlier sensitivity
    return envelope / mvc_val * 100, mvc_val


# Process both recordings, both active channels
for rec in [rec8, rec9]:
    rec['emg_proc'] = {}
    for ch_idx, name in [(0, 'RF'), (1, 'BF')]:
        filt, rect, env = process_emg(rec['ch'][:, ch_idx], fs=FS)
        pct_mvc, mvc_val = normalize_to_mvc(env, rec['time'])
        rec['emg_proc'][name] = {
            'raw': rec['ch'][:, ch_idx],
            'filtered': filt,
            'rectified': rect,
            'envelope': env,
            'pct_mvc': pct_mvc,
            'mvc_val': mvc_val,
        }
    print(f"Processed {Path(rec['path']).name}: RF MVC={rec['emg_proc']['RF']['mvc_val']:.1f}, BF MVC={rec['emg_proc']['BF']['mvc_val']:.1f}")

## Raw Data Overview

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 10), sharex='col')
fig.suptitle('Raw ADC — All 4 Channels', fontsize=13, fontweight='bold')

ch_names = ['Ch0 (RF)', 'Ch1 (BF)', 'Ch2 (Gas — unused)', 'Ch3 (TA — unused)']
colors = ['#2196F3', '#F44336', '#9E9E9E', '#9E9E9E']

for row, (name, color) in enumerate(zip(ch_names, colors)):
    for col, (rec, label) in enumerate([(rec8, 'Rec 8: Stomp → Walk'), (rec9, 'Rec 9: Stomp → Ramp')]):
        ax = axes[row, col]
        ax.plot(rec['time'], rec['ch'][:, row], color=color, lw=0.4, alpha=0.8)
        ax.axhline(4095, color='red', lw=0.8, ls='--', label='ADC max (4095)')
        ax.set_ylabel(name, fontsize=9)
        if row == 0:
            ax.set_title(label, fontweight='bold')
        if row == 3:
            ax.set_xlabel('Time (s)')

axes[0,0].legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()
print("Note: Ch2/Ch3 hit 4095 (ADC saturation) due to electrode issues, not real muscle activity.")

## Processed EMG — Rec 8 (Stomp → Walk)

In [ ]:
rec = rec8
label = 'Rec 8: Stomp (MVC) → Walk'

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.suptitle(f'{label}', fontsize=13, fontweight='bold')

# Panel 1: Filtered EMG
ax = axes[0]
ax.plot(rec['time'], rec['emg_proc']['RF']['filtered'], color='#2196F3', lw=0.5, alpha=0.7, label='RF (Ch0)')
ax.plot(rec['time'], rec['emg_proc']['BF']['filtered'], color='#F44336', lw=0.5, alpha=0.7, label='BF (Ch1)')
ax.set_ylabel('Filtered EMG\n(20–450 Hz bandpass)', fontsize=9)
ax.legend(loc='upper right', fontsize=9)
ax.axvspan(0, 5, alpha=0.08, color='orange', label='MVC window')
ax.axvline(5, color='orange', lw=1, ls='--', label='Stomp end ~5s')

# Panel 2: %MVC envelope
ax = axes[1]
ax.fill_between(rec['time'], rec['emg_proc']['RF']['pct_mvc'], alpha=0.4, color='#2196F3', label='RF')
ax.fill_between(rec['time'], rec['emg_proc']['BF']['pct_mvc'], alpha=0.4, color='#F44336', label='BF')
ax.axhline(100, color='gray', lw=0.8, ls='--', label='100% MVC')
ax.axvspan(0, 5, alpha=0.08, color='orange')
ax.axvline(5, color='orange', lw=1, ls='--')
ax.set_ylabel('EMG Envelope (%MVC)\n[100ms smoothing]', fontsize=9)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, None)

# Panel 3: IMU
ax = axes[2]
# Normalize accel to g (assuming ±2g range -> scale = 2/32768)
# (scale factor unknown without IMU spec, show raw for now)
ax.plot(rec['time'], rec['acc'][:, 0], lw=0.5, color='#4CAF50', alpha=0.7, label='Acc X')
ax.plot(rec['time'], rec['acc'][:, 1], lw=0.5, color='#FF9800', alpha=0.7, label='Acc Y')
ax.plot(rec['time'], rec['acc'][:, 2], lw=0.5, color='#9C27B0', alpha=0.7, label='Acc Z')
ax.set_ylabel('IMU Accel (raw int16)', fontsize=9)
ax.set_xlabel('Time (s)')
ax.legend(loc='upper right', fontsize=9)
ax.axvspan(0, 5, alpha=0.08, color='orange')
ax.axvline(5, color='orange', lw=1, ls='--')

# Annotate regions
for ax in axes:
    ax.text(2.5, ax.get_ylim()[1]*0.85, 'MVC\n(stomp)', ha='center', fontsize=8, color='darkorange')

plt.tight_layout()
plt.show()

## Processed EMG — Rec 9 (Stomp → Ramp)

In [ ]:
rec = rec9
label = 'Rec 9: Stomp (MVC) → Ramp Up/Down (repeated)'

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.suptitle(f'{label}', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(rec['time'], rec['emg_proc']['RF']['filtered'], color='#2196F3', lw=0.5, alpha=0.7, label='RF (Ch0)')
ax.plot(rec['time'], rec['emg_proc']['BF']['filtered'], color='#F44336', lw=0.5, alpha=0.7, label='BF (Ch1)')
ax.set_ylabel('Filtered EMG\n(20–450 Hz bandpass)', fontsize=9)
ax.legend(loc='upper right', fontsize=9)
ax.axvspan(0, 5, alpha=0.08, color='orange')
ax.axvline(5, color='orange', lw=1, ls='--')

ax = axes[1]
ax.fill_between(rec['time'], rec['emg_proc']['RF']['pct_mvc'], alpha=0.4, color='#2196F3', label='RF')
ax.fill_between(rec['time'], rec['emg_proc']['BF']['pct_mvc'], alpha=0.4, color='#F44336', label='BF')
ax.axhline(100, color='gray', lw=0.8, ls='--', label='100% MVC')
ax.axvspan(0, 5, alpha=0.08, color='orange')
ax.axvline(5, color='orange', lw=1, ls='--')
ax.set_ylabel('EMG Envelope (%MVC)\n[100ms smoothing]', fontsize=9)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, None)

ax = axes[2]
ax.plot(rec['time'], rec['acc'][:, 0], lw=0.5, color='#4CAF50', alpha=0.7, label='Acc X')
ax.plot(rec['time'], rec['acc'][:, 1], lw=0.5, color='#FF9800', alpha=0.7, label='Acc Y')
ax.plot(rec['time'], rec['acc'][:, 2], lw=0.5, color='#9C27B0', alpha=0.7, label='Acc Z')
ax.set_ylabel('IMU Accel (raw int16)', fontsize=9)
ax.set_xlabel('Time (s)')
ax.legend(loc='upper right', fontsize=9)
ax.axvspan(0, 5, alpha=0.08, color='orange')
ax.axvline(5, color='orange', lw=1, ls='--')

for ax in axes:
    ax.text(2.5, ax.get_ylim()[1]*0.85, 'MVC\n(stomp)', ha='center', fontsize=8, color='darkorange')

plt.tight_layout()
plt.show()

## Side-by-Side Comparison: Walking vs. Ramp
Only the **locomotion phase** (after stomp MVC), both channels

In [ ]:
# Trim to locomotion phase (post-MVC, starting ~5s)
STOMP_END = 5.0

def get_loco_phase(rec, start=STOMP_END):
    mask = rec['time'] >= start
    t = rec['time'][mask] - start
    return t, mask

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex='col', sharey='row')
fig.suptitle('EMG During Locomotion (post-MVC, normalized to %MVC)', fontsize=13, fontweight='bold')

for col, (rec, title) in enumerate([(rec8, 'Walk'), (rec9, 'Ramp (up/down)')]):
    t, mask = get_loco_phase(rec)
    for row, muscle in enumerate(['RF', 'BF']):
        ax = axes[row, col]
        pct = rec['emg_proc'][muscle]['pct_mvc'][mask]
        color = '#2196F3' if muscle == 'RF' else '#F44336'
        ax.fill_between(t, pct, alpha=0.4, color=color)
        ax.plot(t, pct, color=color, lw=0.8, label=muscle)
        ax.axhline(100, color='gray', lw=0.8, ls='--', alpha=0.6)
        if row == 0:
            ax.set_title(title, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'{muscle} (%MVC)', fontsize=10)
        if row == 1:
            ax.set_xlabel('Time since locomotion start (s)')
        ax.set_ylim(0, None)

plt.tight_layout()
plt.show()

## IMU Analysis — Gait Detection
Gyro Z or vertical accel can be used to detect step cycles

In [ ]:
# Focus on locomotion phase, look for periodicity in IMU
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex='col')
fig.suptitle('IMU — Gyroscope During Locomotion', fontsize=13, fontweight='bold')

gyro_labels = ['Gyro X', 'Gyro Y', 'Gyro Z']
gyro_colors = ['#E91E63', '#00BCD4', '#8BC34A']

for col, (rec, title) in enumerate([(rec8, 'Walk'), (rec9, 'Ramp')]):
    t, mask = get_loco_phase(rec)
    ax_gyro = axes[0, col]
    for i in range(3):
        ax_gyro.plot(t, rec['gyr'][mask, i], lw=0.6, alpha=0.8,
                     color=gyro_colors[i], label=gyro_labels[i])
    ax_gyro.set_title(title, fontweight='bold')
    ax_gyro.set_ylabel('Gyro (raw int16)')
    ax_gyro.legend(fontsize=8)

    ax_acc = axes[1, col]
    acc_colors = ['#4CAF50', '#FF9800', '#9C27B0']
    for i in range(3):
        ax_acc.plot(t, rec['acc'][mask, i], lw=0.6, alpha=0.8,
                    color=acc_colors[i], label=f'Acc {"XYZ"[i]}')
    ax_acc.set_ylabel('Accel (raw int16)')
    ax_acc.set_xlabel('Time (s)')
    ax_acc.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Step detection via accel magnitude peaks
from scipy.signal import find_peaks

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
fig.suptitle('Step Detection (Accel Magnitude)', fontsize=13, fontweight='bold')

for ax, (rec, title) in zip(axes, [(rec8, 'Rec 8 — Walk'), (rec9, 'Rec 9 — Ramp')]):
    t, mask = get_loco_phase(rec)
    acc_seg = rec['acc'][mask].astype(float)
    
    # Accel magnitude
    mag = np.linalg.norm(acc_seg, axis=1)
    
    # Smooth for step detection
    b, a = signal.butter(4, 10/250, btype='low')  # 10 Hz LP
    mag_smooth = signal.filtfilt(b, a, mag)
    
    # Find peaks (steps)
    # min distance: ~0.4s between steps at 500Hz = 200 samples
    peaks, _ = find_peaks(mag_smooth, distance=int(0.35*FS),
                           height=np.percentile(mag_smooth, 60))
    
    ax.plot(t, mag_smooth, lw=0.8, color='#607D8B', label='|Accel| (smoothed)')
    ax.plot(t[peaks], mag_smooth[peaks], 'rv', markersize=6, label=f'Steps detected: {len(peaks)}')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Accel Magnitude (raw)')
    ax.set_xlabel('Time since locomotion start (s)')
    ax.legend(fontsize=9)
    
    if len(peaks) > 2:
        step_intervals = np.diff(t[peaks])
        cadence = 60 / step_intervals.mean()
        print(f"{title}: {len(peaks)} steps detected, mean cadence ≈ {cadence:.0f} steps/min")

plt.tight_layout()
plt.show()

## Frequency Analysis — EMG Power Spectrum

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey='row')
fig.suptitle('EMG Power Spectral Density — Locomotion Phase', fontsize=13, fontweight='bold')

for col, (rec, title) in enumerate([(rec8, 'Walk'), (rec9, 'Ramp')]):
    _, mask = get_loco_phase(rec)
    for row, muscle in enumerate(['RF', 'BF']):
        ax = axes[row, col]
        seg = rec['emg_proc'][muscle]['filtered'][mask]
        f, psd = signal.welch(seg, fs=FS, nperseg=512)
        ax.semilogy(f, psd, lw=1.2,
                    color='#2196F3' if muscle == 'RF' else '#F44336')
        ax.axvline(20, color='gray', ls='--', lw=0.8, label='20 Hz')
        ax.axvline(200, color='gray', ls=':', lw=0.8, label='200 Hz')
        
        # Median frequency
        cumulative = np.cumsum(psd)
        mf = f[np.searchsorted(cumulative, cumulative[-1]/2)]
        ax.axvline(mf, color='black', ls='-', lw=1, alpha=0.7, label=f'Median f: {mf:.0f} Hz')
        
        ax.set_xlim(0, 499)
        ax.set_xlabel('Frequency (Hz)')
        if col == 0:
            ax.set_ylabel(f'{muscle} PSD')
        if row == 0:
            ax.set_title(title, fontweight='bold')
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY — Locomotion Phase Only (post-5s MVC)")
print("=" * 60)

for rec, task in [(rec8, 'Rec 8 — Walk'), (rec9, 'Rec 9 — Ramp')]:
    _, mask = get_loco_phase(rec)
    loco_dur = rec['time'][mask][-1] - STOMP_END
    print(f"\n{task}  ({loco_dur:.1f}s of locomotion)")
    print(f"  {'Muscle':<8} {'Mean %MVC':>10} {'Peak %MVC':>10} {'RMS':>8}")
    for muscle in ['RF', 'BF']:
        pct = rec['emg_proc'][muscle]['pct_mvc'][mask]
        filt = rec['emg_proc'][muscle]['filtered'][mask]
        rms = np.sqrt(np.mean(filt**2))
        print(f"  {muscle:<8} {pct.mean():>9.1f}% {np.percentile(pct,95):>9.1f}% {rms:>8.1f}")

print("\n" + "=" * 60)
print("Notes:")
print("  - MVC estimated from 95th percentile of stomp window (0–5s)")
print("  - Ch2/Ch3 (Gas/TA) excluded — ADC saturation artifacts")
print("  - IMU scale factors unknown (shown as raw int16)")
print("  - Motor data not yet available")

## TODO / Next Steps

- [ ] **Confirm byte layout with Navid** — especially IMU scale factors (accel g/LSB, gyro deg/s/LSB)
- [ ] **Fix Ch2/Ch3 saturation** — extend cable for Gas/TA, or confirm electrode issue
- [ ] **Add motor torque data** when Navid adds it to the format (new field after byte 29?)
- [ ] **Manual phase labeling** — mark stomp end, uphill start, downhill start in Rec 9
- [ ] **Gait cycle segmentation** — use IMU peaks for phase-averaged EMG plots
- [ ] **Cross-recording MVC normalization** — use a single dedicated MVC trial rather than per-recording stomp